# YOLO를 활용한 객체 추적 + 투기 행위 탐지

## 드라이브

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


## 데이터셋 압축 해제

In [ ]:

from google.colab import files
import zipfile
import os

# 2. 압축 파일 경로 설정 (본인의 파일 경로에 맞게 수정!)
zip_path = "/content/drive/MyDrive/Roboflow_YOLO.zip"

# 3. 압축을 풀 폴더 생성
extract_folder = "/content/unzipped_data"
os.makedirs(extract_folder, exist_ok=True)

# 4. ZIP 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print(f"✅ 압축 해제 완료! → 위치: {extract_folder}")



✅ 압축 해제 완료! → 위치: /content/unzipped_data


## 동영상 기준으로 Train:Valid:Test 로 분할

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from collections import defaultdict
import yaml

# ✅ 경로 설정
image_dir = '/content/unzipped_data/Roboflow_YOLO/train/images'
label_dir = '/content/unzipped_data/Roboflow_YOLO/train/labels'
src_yaml  = '/content/unzipped_data/Roboflow_YOLO/data.yaml'  # 원본 yaml
output_dir = 'split_dataset'  # 결과 저장 경로

# ✅ 비율 설정
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# ✅ 영상 단위 그룹핑
video_groups = defaultdict(list)
for fname in os.listdir(image_dir):
    if fname.lower().endswith('.jpg') and '_frame_' in fname:
        video_id = fname.split('_frame_')[0]
        video_groups[video_id].append(fname)

print(f"총 영상 수: {len(video_groups)}")

# ✅ 영상 단위로 split
video_ids = list(video_groups.keys())
if len(video_ids) < 3:
    raise ValueError(f"영상 수가 너무 적습니다. 현재 영상 개수: {len(video_ids)}")

train_ids, temp_ids = train_test_split(video_ids, test_size=(1 - train_ratio), random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=42)

splits = {
    'train': train_ids,
    'val': val_ids,  # valid 그대로 사용
    'test': test_ids
}

# ✅ Roboflow 스타일 디렉터리 생성
for split in splits:
    os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)

# ✅ 파일 복사
for split, video_list in splits.items():
    for vid in video_list:
        for img_fname in video_groups[vid]:
            label_fname = img_fname.replace('.jpg', '.txt')

            img_src = os.path.join(image_dir, img_fname)
            lbl_src = os.path.join(label_dir, label_fname)

            img_dst = os.path.join(output_dir, split, 'images', img_fname)
            lbl_dst = os.path.join(output_dir, split, 'labels', label_fname)

            shutil.copy(img_src, img_dst)
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, lbl_dst)

print("✅ Roboflow 형식으로 split 완료! (train/, valid/, test/ 내부에 images/labels 생성)")

# ✅ data.yaml 복사 + 경로 업데이트
with open(src_yaml, 'r') as f:
    orig_yaml = yaml.safe_load(f)

names = orig_yaml.get('names')
nc = orig_yaml.get('nc', len(names) if names is not None else None)

new_yaml = {
    'path': None,
    'train': os.path.abspath(os.path.join(output_dir, 'train', 'images')),
    'val': os.path.abspath(os.path.join(output_dir, 'val', 'images')),  # valid 그대로
    'test': os.path.abspath(os.path.join(output_dir, 'test', 'images')),
    'names': names,
    'nc': nc
}

dst_yaml_path = os.path.join(output_dir, 'data.yaml')
with open(dst_yaml_path, 'w') as f:
    yaml.dump(new_yaml, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml 생성 완료: {dst_yaml_path}")


총 영상 수: 246
✅ Roboflow 형식으로 split 완료! (train/, valid/, test/ 내부에 images/labels 생성)
✅ data.yaml 생성 완료: split_dataset/data.yaml


## can, box 등등 쓰레기 라벨링 한 것들 -> trash로 통합

In [ ]:
# Colab에서 실행
import os, re, shutil, sys, subprocess
from pathlib import Path

# pyyaml 설치
try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml"])
    import yaml

# ===== 설정 =====
INPUT_DIR  = Path("/content/split_dataset")          # 원본 데이터셋(root에 train/ val(or valid)/ test)
OUTPUT_DIR = Path("/content/split_dataset_merged")   # 결과 데이터셋(새로 생성)
WRITE_ABSOLUTE_PATHS = True                          # data.yaml에 절대경로 쓰기(권장)

# 병합 규칙
NEW_NAMES = ['car','car_door','hand','human','license_plate','trash']
KEEP      = ['car','car_door','hand','human','license_plate']
MERGE2TR  = ['box','can','plastic','plastic_bag','trash']  # → 'trash'

# ===== 유틸 =====
FALLBACK_ORIG = ['box','can','car','car_door','hand','human','license_plate','plastic','plastic_bag','trash']

def load_orig_names(src: Path):
    for fn in ("data.yaml","data.yml"):
        p = src / fn
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                d = yaml.safe_load(f) or {}
            if isinstance(d.get("names"), list) and d["names"]:
                return d["names"]
    return FALLBACK_ORIG

def make_mapping(orig_names, new_names):
    oidx = {n:i for i,n in enumerate(orig_names)}
    nidx = {n:i for i,n in enumerate(new_names)}
    m = {}
    for n in KEEP:
        if n in oidx: m[oidx[n]] = nidx[n]
    for n in MERGE2TR:
        if n in oidx: m[oidx[n]] = nidx['trash']
    return m

INT = re.compile(r"^-?\d+$")
def remap_line(line: str, mapping: dict):
    s = line.strip()
    if not s or s.startswith("#"):
        return line
    parts = s.split()
    if not parts or not INT.match(parts[0]):
        return line
    old_cls = int(parts[0])
    if old_cls in mapping:
        parts[0] = str(mapping[old_cls])
        return " ".join(parts) + ("\n" if line.endswith("\n") else "")
    return line

def copy_images(src: Path, dst: Path):
    dst.mkdir(parents=True, exist_ok=True)
    if src.exists():
        # 폴더 구조 보존 복사
        for p in src.rglob("*"):
            if p.is_file():
                out = dst / p.relative_to(src)
                out.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, out)

def process_labels(src: Path, dst: Path, mapping: dict):
    dst.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        return (0,0,0)
    total = ch_files = ch_lines = 0
    for fp in sorted(src.glob("*.txt")):
        total += 1
        with open(fp, "r", encoding="utf-8") as f:
            lines = f.readlines()
        out, changed = [], False
        for ln in lines:
            nl = remap_line(ln, mapping)
            if nl != ln:
                ch_lines += 1; changed = True
            out.append(nl)
        if changed: ch_files += 1
        with open(dst / fp.name, "w", encoding="utf-8") as f:
            f.writelines(out)
    return (total, ch_files, ch_lines)

def write_yaml(out_dir: Path, absolute: bool):
    if absolute:
        train_p = (out_dir/"train/images").resolve().as_posix()
        val_p   = (out_dir/"val/images").resolve().as_posix()
        test_p  = (out_dir/"test/images").resolve().as_posix()
    else:
        train_p, val_p, test_p = "train/images", "val/images", "test/images"

    data = {
        "train": train_p,
        "val":   val_p,
        "test":  test_p,
        "nc":    len(NEW_NAMES),
        "names": NEW_NAMES,
    }
    with open(out_dir / "data.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True, default_flow_style=False)

# ===== 실행 =====
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

orig_names = load_orig_names(INPUT_DIR)
mapping = make_mapping(orig_names, NEW_NAMES)
print("원본 names:", orig_names)
print("Old→New mapping:", mapping)

# 'valid' → 'val' 표준화, 없으면 빈 폴더도 생성
split_map = [("train","train"), ("val","val"),  ("test","test")]
present = set()
for src_name, out_name in split_map:
    img_src = INPUT_DIR / src_name / "images"
    lbl_src = INPUT_DIR / src_name / "labels"
    img_dst = OUTPUT_DIR / out_name / "images"
    lbl_dst = OUTPUT_DIR / out_name / "labels"

    # images
    copy_images(img_src, img_dst)
    # labels
    t, cf, cl = process_labels(lbl_src, lbl_dst, mapping)
    present.add(out_name)
    print(f"[{src_name} → {out_name}] 라벨 파일 {t}개, 변경 파일 {cf}개, 변경 라인 {cl}줄")

# 없던 split도 폴더는 만들어서 경로 보장
for sp in ("train","val","test"):
    (OUTPUT_DIR/sp/"images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR/sp/"labels").mkdir(parents=True, exist_ok=True)

write_yaml(OUTPUT_DIR, WRITE_ABSOLUTE_PATHS)

print("\n✅ 완료!")
print("새 데이터셋:", OUTPUT_DIR)
print("data.yaml:", (OUTPUT_DIR/"data.yaml").resolve())
print("예:", (OUTPUT_DIR/"test/images").resolve())


원본 names: ['box', 'can', 'car', 'car_door', 'hand', 'human', 'license_plate', 'plastic', 'plastic_bag', 'trash']
Old→New mapping: {2: 0, 3: 1, 4: 2, 5: 3, 6: 4, 0: 5, 1: 5, 7: 5, 8: 5, 9: 5}
[train → train] 라벨 파일 5131개, 변경 파일 5124개, 변경 라인 20287줄
[val → val] 라벨 파일 1397개, 변경 파일 1397개, 변경 라인 3807줄
[test → test] 라벨 파일 723개, 변경 파일 722개, 변경 라인 3328줄

✅ 완료!
새 데이터셋: /content/split_dataset_merged
data.yaml: /content/split_dataset_merged/data.yaml
예: /content/split_dataset_merged/test/images


## YOLO 모델

In [ ]:
!pip install ultralytics roboflow tqdm opencv-python pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 M

In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split
from collections import defaultdict
import yaml

# ✅ 경로 설정
image_dir = '/content/unzipped_data/Roboflow_YOLO/train/images'
label_dir = '/content/unzipped_data/Roboflow_YOLO/train/labels'
src_yaml  = '/content/unzipped_data/Roboflow_YOLO/data.yaml'  # 원본 yaml
output_dir = 'split_dataset'  # 결과 저장 경로

# ✅ 비율 설정
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# ✅ 영상 단위 그룹핑
video_groups = defaultdict(list)
for fname in os.listdir(image_dir):
    if fname.lower().endswith('.jpg') and '_frame_' in fname:
        video_id = fname.split('_frame_')[0]
        video_groups[video_id].append(fname)

print(f"총 영상 수: {len(video_groups)}")

# ✅ 영상 단위로 split
video_ids = list(video_groups.keys())
if len(video_ids) < 3:
    raise ValueError(f"영상 수가 너무 적습니다. 현재 영상 개수: {len(video_ids)}")

train_ids, temp_ids = train_test_split(video_ids, test_size=(1 - train_ratio), random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=(test_ratio / (val_ratio + test_ratio)), random_state=42)

splits = {
    'train': train_ids,
    'val': val_ids,  # valid 그대로 사용
    'test': test_ids
}

# ✅ Roboflow 스타일 디렉터리 생성
for split in splits:
    os.makedirs(os.path.join(output_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'labels'), exist_ok=True)

# ✅ 파일 복사
for split, video_list in splits.items():
    for vid in video_list:
        for img_fname in video_groups[vid]:
            label_fname = img_fname.replace('.jpg', '.txt')

            img_src = os.path.join(image_dir, img_fname)
            lbl_src = os.path.join(label_dir, label_fname)

            img_dst = os.path.join(output_dir, split, 'images', img_fname)
            lbl_dst = os.path.join(output_dir, split, 'labels', label_fname)

            shutil.copy(img_src, img_dst)
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, lbl_dst)

print("✅ Roboflow 형식으로 split 완료! (train/, valid/, test/ 내부에 images/labels 생성)")

# ✅ data.yaml 복사 + 경로 업데이트
with open(src_yaml, 'r') as f:
    orig_yaml = yaml.safe_load(f)

names = orig_yaml.get('names')
nc = orig_yaml.get('nc', len(names) if names is not None else None)

new_yaml = {
    'path': None,
    'train': os.path.abspath(os.path.join(output_dir, 'train', 'images')),
    'valid': os.path.abspath(os.path.join(output_dir, 'valid', 'images')),  # valid 그대로
    'test': os.path.abspath(os.path.join(output_dir, 'test', 'images')),
    'names': names,
    'nc': nc
}

dst_yaml_path = os.path.join(output_dir, 'data.yaml')
with open(dst_yaml_path, 'w') as f:
    yaml.dump(new_yaml, f, default_flow_style=False, sort_keys=False)

print(f"✅ data.yaml 생성 완료: {dst_yaml_path}")


### 오버 샘플링 코드


In [ ]:
import os, random, glob, shutil, math
from collections import Counter, defaultdict
import yaml
import cv2
import numpy as np
from tqdm import tqdm

# -----------------------------
# 설정값
# -----------------------------
DATA_YAML = "/content/split_dataset_merged/data.yaml"   # 기존 data.yaml
OUT_DIR = "/content/oversampled_dataset"     # 결과 데이터셋 루트
MAX_DUP = 1                                      # 이미지 중복 최대 배수(오버샘플)
ALPHA = 0.3                                      # oversample 강도 (0.3 ~ 0.5로 낮춤)
CP_PER_IMAGE = 1                                  # 소수클래스 포함 이미지당 copy-paste 생성 수
PASTE_SCALE_RANGE = (0.9, 1.1)                    # 붙여넣기 시 스케일 랜덤 범위
PASTE_JITTER = 0.05                               # 붙여넣기 좌표 소폭 무작위 (이미지 크기 비율)
RAND_SEED = 42
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)

IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

def save_yaml(obj, path):
    with open(path, "w") as f:
        yaml.safe_dump(obj, f, sort_keys=False, allow_unicode=True)

def yolo_to_xyxy(box, W, H):
    cls, xc, yc, w, h = box
    x1 = (xc - w/2) * W
    y1 = (yc - h/2) * H
    x2 = (xc + w/2) * W
    y2 = (yc + h/2) * H
    return int(cls), max(0, int(x1)), max(0, int(y1)), min(W-1, int(x2)), min(H-1, int(y2))

def xyxy_to_yolo(cls, x1, y1, x2, y2, W, H):
    w = (x2 - x1) / W
    h = (y2 - y1) / H
    xc = (x1 + x2) / (2 * W)
    yc = (y1 + y2) / (2 * H)
    return [cls, xc, yc, w, h]

def read_labels(lbl_path):
    if not os.path.exists(lbl_path):
        return []
    lines = open(lbl_path).read().strip().splitlines()
    boxes = []
    for ln in lines:
        if not ln.strip():
            continue
        parts = ln.strip().split()
        cls = int(parts[0]); xc, yc, w, h = map(float, parts[1:5])
        boxes.append([cls, xc, yc, w, h])
    return boxes

def write_labels(lbl_path, boxes):
    with open(lbl_path, "w") as f:
        for b in boxes:
            f.write(f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")

def list_images(dir_or_txt):
    if os.path.isfile(dir_or_txt) and dir_or_txt.endswith(".txt"):
        return [p for p in open(dir_or_txt).read().splitlines() if p.strip()]
    paths = []
    for ex in IMG_EXTS:
        paths += glob.glob(os.path.join(dir_or_txt, "**", ex), recursive=True)
    return sorted(paths)

def infer_label_path(img_path, root_img, root_lbl):
    rel = os.path.relpath(img_path, root_img)
    lbl = os.path.splitext(rel)[0] + ".txt"
    return os.path.join(root_lbl, lbl)

def make_dir(p):
    os.makedirs(p, exist_ok=True)

# -----------------------------
# 1) 데이터 로드 & 분포 집계
# -----------------------------
cfg = load_yaml(DATA_YAML)
train_src = cfg["train"]
val_src = cfg.get("val", None)
names = cfg.get("names", None)

train_imgs = list_images(train_src)
if len(train_imgs) == 0:
    raise FileNotFoundError(f"train 경로에 이미지가 없습니다: {train_src}")

# 이미지/라벨 루트 추정
def guess_roots(example_img):
    parts = example_img.split(os.sep)
    if "images" in parts:
        idx = parts.index("images")
        img_root = os.sep.join(parts[:idx+1])
        lbl_root = os.sep.join(parts[:idx] + ["labels"])
        return img_root, lbl_root
    return os.path.dirname(example_img), os.path.dirname(example_img)

img_root, lbl_root = guess_roots(train_imgs[0])

class_counts = Counter()
img_classes = {}

for img_path in tqdm(train_imgs, desc="Counting labels"):
    H, W = None, None
    lbl_path = infer_label_path(img_path, img_root, lbl_root)
    boxes = read_labels(lbl_path)
    cls_set = set()
    for b in boxes:
        class_counts[int(b[0])] += 1
        cls_set.add(int(b[0]))
    img_classes[img_path] = cls_set

if not class_counts:
    raise RuntimeError("라벨이 비어 있습니다. *.txt 라벨 파일을 확인하세요.")

max_count = max(class_counts.values())
minority = {c for c, k in class_counts.items() if k < 0.5 * max_count}

print("Class counts:", class_counts)
print("Minority classes:", minority)

# -----------------------------
# 2) 증강 비율을 맞추기 위한 오버샘플링
# -----------------------------
original_count = len(train_imgs)
target_count = math.ceil(original_count * 1.5)  # 1.5배 증강 목표
augmented_images_needed = target_count - original_count

# 소수 클래스만 오버샘플링하여 증강 비율 맞추기
dup_list = []
for img_path, cls_set in img_classes.items():
    if not cls_set:
        dup = 1
    else:
        if cls_set & minority:
            # 소수 클래스가 포함된 경우에만 duplication factor 계산
            min_freq = min(class_counts[c] for c in cls_set)
            target = max_count
            ratio = target / max(min_freq, 1)
            dup = int(min(MAX_DUP, math.ceil(ratio ** ALPHA)))
        else:
            dup = 1
    dup = max(1, dup)
    dup_list += [img_path] * dup

# 필요한 만큼만 증강
dup_list = dup_list[:augmented_images_needed]
random.shuffle(dup_list)

# -----------------------------
# 3) Copy-Paste augmentation (소수 클래스만)
# -----------------------------
aug_img_dir = os.path.join(OUT_DIR, "images", "train")
aug_lbl_dir = os.path.join(OUT_DIR, "labels", "train")
make_dir(aug_img_dir); make_dir(aug_lbl_dir)

for img_path in tqdm(train_imgs, desc="Copy original to OUT_DIR"):
    rel = os.path.relpath(img_path, img_root) if "images" in img_path else os.path.basename(img_path)
    dst_img = os.path.join(aug_img_dir, rel)
    os.makedirs(os.path.dirname(dst_img), exist_ok=True)
    shutil.copy2(img_path, dst_img)

    lbl_src = infer_label_path(img_path, img_root, lbl_root)
    if not os.path.exists(lbl_src):
        lbl_src = os.path.splitext(img_path)[0] + ".txt"
    rel_lbl = os.path.relpath(lbl_src, lbl_root) if os.path.exists(lbl_src) and ("labels" in lbl_src) else os.path.basename(lbl_src)
    dst_lbl = os.path.join(aug_lbl_dir, rel_lbl)
    os.makedirs(os.path.dirname(dst_lbl), exist_ok=True)
    if os.path.exists(lbl_src):
        shutil.copy2(lbl_src, dst_lbl)
    else:
        open(dst_lbl, "w").close()

# copy-paste를 위한 인덱스
all_bg_imgs = list_images(aug_img_dir)
bg_index = [p for p in all_bg_imgs]

def paste_box(src_img, box_xyxy, dst_img, dst_boxes, cls_id):
    Hd, Wd = dst_img.shape[:2]
    x1, y1, x2, y2 = box_xyxy

    # 1) 원본에서 잘라오기
    crop = src_img[max(0,y1):max(0,y2), max(0,x1):max(0,x2)]
    if crop.size == 0:
        return dst_img, dst_boxes
    ch = 1 if len(crop.shape) == 2 else crop.shape[2]
    if ch == 1:
        crop = cv2.cvtColor(crop, cv2.COLOR_GRAY2BGR)

    # 2) 스케일 결정 (아직 resize 하지 않음)
    scale = random.uniform(*PASTE_SCALE_RANGE)
    base_w = max(2, int((x2 - x1) * scale))
    base_h = max(2, int((y2 - y1) * scale))

    # 3) 위치 결정 + 지터
    dx = random.randint(0, max(0, Wd - 2))
    dy = random.randint(0, max(0, Hd - 2))
    dx = int(np.clip(dx + random.randint(int(-PASTE_JITTER*Wd), int(PASTE_JITTER*Wd)), 0, Wd - 2))
    dy = int(np.clip(dy + random.randint(int(-PASTE_JITTER*Hd), int(PASTE_JITTER*Hd)), 0, Hd - 2))

    # 4) 대상 이미지 경계에 맞춰 최종 크기 clamp
    final_w = min(base_w, Wd - dx)
    final_h = min(base_h, Hd - dy)
    if final_w < 2 or final_h < 2:
        return dst_img, dst_boxes

    # 5) **최종 크기로 단 한 번만 resize**
    crop_resized = cv2.resize(crop, (final_w, final_h), interpolation=cv2.INTER_LINEAR)

    # 6) 붙여넣기
    dst_img[dy:dy+final_h, dx:dx+final_w] = crop_resized

    # 7) 라벨 추가 (YOLO)
    nx1, ny1, nx2, ny2 = dx, dy, dx + final_w, dy + final_h
    dst_boxes.append(xyxy_to_yolo(cls_id, nx1, ny1, nx2, ny2, Wd, Hd))

    return dst_img, dst_boxes


print("Generating copy-paste aug...")

# 이미지 읽을 때 W, H 값을 가져오도록 수정
for img_path in tqdm(train_imgs):
    lbl_src = infer_label_path(img_path, img_root, lbl_root)
    if not os.path.exists(lbl_src):
        lbl_src = os.path.splitext(img_path)[0] + ".txt"
    boxes = read_labels(lbl_src)

    # 이미지 크기 가져오기
    src_img = cv2.imread(img_path)
    if src_img is None:
        continue
    H, W = src_img.shape[:2]  # 이미지의 높이와 너비를 H, W로 설정

    min_boxes_xyxy = []
    for b in boxes:
        cls = int(b[0])
        if cls in minority:
            _, x1, y1, x2, y2 = yolo_to_xyxy(b, W, H)  # W, H 전달
            if (x2 - x1) >= 4 and (y2 - y1) >= 4:
                min_boxes_xyxy.append((cls, (x1, y1, x2, y2)))

    if not min_boxes_xyxy:
        continue

    for n in range(CP_PER_IMAGE):
        bg_img_path = random.choice(bg_index)
        bg = cv2.imread(bg_img_path)
        if bg is None:
            continue
        Hd, Wd = bg.shape[:2]  # 배경 이미지 크기
        bg_lbl_rel = os.path.relpath(bg_img_path, aug_img_dir)
        bg_lbl_path = os.path.join(aug_lbl_dir, os.path.splitext(bg_lbl_rel)[0] + ".txt")
        bg_boxes = read_labels(bg_lbl_path)

        num_paste = random.randint(1, min(3, len(min_boxes_xyxy)))
        paste_samples = random.sample(min_boxes_xyxy, num_paste)
        new_img = bg.copy()
        new_boxes = bg_boxes.copy()

        for cls_id, (x1, y1, x2, y2) in paste_samples:
            new_img, new_boxes = paste_box(src_img, (x1, y1, x2, y2), new_img, new_boxes, cls_id)

        base, ext = os.path.splitext(bg_lbl_rel)
        out_stub = f"{base}_cp{n:02d}"
        out_img_path = os.path.join(aug_img_dir, out_stub + ".jpg")
        out_lbl_path = os.path.join(aug_lbl_dir, out_stub + ".txt")
        os.makedirs(os.path.dirname(out_img_path), exist_ok=True)
        cv2.imwrite(out_img_path, new_img)
        write_labels(out_lbl_path, new_boxes)



# -----------------------------
# 4) 오버샘플링된 train.txt 저장
# -----------------------------
all_train_imgs_after_aug = list_images(aug_img_dir)

new_img_classes = {}
new_counts = Counter()
for img_path in tqdm(all_train_imgs_after_aug, desc="Recount after aug"):
    rel = os.path.relpath(img_path, aug_img_dir)
    lbl_path = os.path.join(aug_lbl_dir, os.path.splitext(rel)[0] + ".txt")
    boxes = read_labels(lbl_path)
    cls_set = set()
    for b in boxes:
        new_counts[int(b[0])] += 1
        cls_set.add(int(b[0]))
    new_img_classes[img_path] = cls_set

max_count2 = max(new_counts.values()) if new_counts else 1

oversampled_list = []
for img_path, cls_set in new_img_classes.items():
    if not cls_set:
        dup = 1
    else:
        min_freq = min(new_counts[c] for c in cls_set) if cls_set else max_count2
        ratio = max_count2 / max(min_freq, 1)
        dup = int(min(MAX_DUP, math.ceil(ratio ** ALPHA)))
    dup = max(1, dup)
    oversampled_list += [img_path] * dup

random.shuffle(oversampled_list)

train_txt_path = os.path.join(OUT_DIR, "train.txt")
with open(train_txt_path, "w") as f:
    for p in oversampled_list:
        f.write(p + "\n")

print(f"Saved oversampled train list: {train_txt_path} ({len(oversampled_list)} lines)")

# -----------------------------
# 5) 새 data.yaml 저장
# -----------------------------
new_yaml = dict(cfg)  # shallow copy
new_yaml["train"] = train_txt_path
NEW_DATA_YAML = os.path.join(OUT_DIR, "data_oversampled.yaml")
save_yaml(new_yaml, NEW_DATA_YAML)
print(f"Saved new data.yaml: {NEW_DATA_YAML}")


Counting labels: 100%|██████████| 5346/5346 [00:00<00:00, 17389.53it/s]


Class counts: Counter({0: 10769, 4: 4682, 5: 2130, 3: 1834, 1: 1412, 2: 405})
Minority classes: {1, 2, 3, 4, 5}


Copy original to OUT_DIR: 100%|██████████| 5346/5346 [00:07<00:00, 684.36it/s] 


Generating copy-paste aug...


Recount after aug: 100%|██████████| 10187/10187 [00:01<00:00, 7577.38it/s]

Saved oversampled train list: /content/oversampled_dataset/train.txt (10187 lines)
Saved new data.yaml: /content/oversampled_dataset/data_oversampled.yaml


# 최종 YOLO 훈련 코드

In [ ]:
from ultralytics import YOLO

# ✅ 사전 학습된 YOLO 모델 불러오기
model = YOLO('yolo11l.pt')  # 또는 yolov8s.pt, yolov8m.pt


# ✅ 하이퍼파라미터 튜닝 설정
model.train(
    data='/content/split_dataset_merged/data.yaml',  # 데이터셋 yaml 경로
    epochs=150,                      # 학습 epoch 수 (기본: 100)
    imgsz=640,                       # 입력 이미지 크기
    batch=16,                        # 배치 크기 (GPU 메모리에 따라 조정)
    name='dumping_detector_v2',     # 결과 저장 폴더 이름
    pretrained=True,                # 사전학습된 가중치 사용
    lr0=0.001,                      # 초기 학습률 (기본: 0.01)
    lrf=0.01,                       # 최종 학습률 (lr0에서 얼마나 줄일지)
    momentum=0.937,                 # 모멘텀
    weight_decay=0.0005,            # 가중치 감소 (정규화)
    warmup_epochs=3.0,              # 워밍업 epoch 수
    warmup_momentum=0.8,            # 워밍업 중 모멘텀
    warmup_bias_lr=0.1,             # 워밍업 중 바이어스 학습률
    box=7.5,                        # 박스 회귀 손실 가중치
    cls=0.5,                        # 클래스 손실 가중치
    dfl=1.5,                        # DFL 손실 가중치 (Distribution Focal Loss)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,  # HSV 색상 데이터 증강
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0,  # 기타 증강 파라미터
    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0,     # 수직/수평 뒤집기, 모자이크, 믹스업
    patience=15                     # Early stopping 기준
)

Ultralytics 8.3.176 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/split_dataset_merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=dumping_detector_v25, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=20, perspective=0.0, plots=

train: Scanning /content/split_dataset_merged/train/labels... 5221 images, 7 backgrounds, 0 corrupt: 100%|██████████| 5221/5221 [00:12<00:00, 409.53it/s]

train: New cache created: /content/split_dataset_merged/train/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 171, len(boxes) = 19474. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1253.8±1115.3 MB/s, size: 253.4 KB)


val: Scanning /content/split_dataset_merged/val/labels.cache... 1305 images, 1 backgrounds, 0 corrupt: 100%|██████████| 1305/1305 [00:00<?, ?it/s]

...

                   all       1305       5233      0.665      0.581       0.58      0.293



100 epochs completed in 7.399 hours.
Optimizer stripped from runs/detect/dumping_detector_v25/weights/last.pt, 51.2MB
Optimizer stripped from runs/detect/dumping_detector_v25/weights/best.pt, 51.2MB

Validating runs/detect/dumping_detector_v25/weights/best.pt...
Ultralytics 8.3.176 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11l summary (fused): 190 layers, 25,283,938 parameters, 0 gradients, 86.6 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 41/41 [00:22<00:00,  1.85it/s]


                   all       1305       5233      0.699      0.572      0.581      0.296
                   car       1292       2718      0.905      0.939      0.945      0.661
              car_door        344        344      0.804      0.942      0.938      0.504
                  hand         74         82      0.425      0.122      0.116     0.0325
                 human        354        370       0.82      0.627      0.681      0.279
         license_plate        676       1212      0.592      0.387      0.409      0.157
                 trash        499        507      0.647      0.418        0.4      0.141
Speed: 0.2ms preprocess, 7.5ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/dumping_detector_v25


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e9dd6f1ca50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

# RT-DETR

In [ ]:
import os, random, glob, shutil, math
from collections import Counter, defaultdict
import yaml
import cv2
import numpy as np
from tqdm import tqdm

# =============================
# 설정값
# =============================
DATA_YAML = "/content/split_dataset_merged/data.yaml"   # 기존 data.yaml
OUT_DIR = "/content/oversampled_dataset"                # 결과 데이터셋 루트
MAX_DUP = 1                                             # 이미지 중복 최대 배수(오버샘플)
ALPHA = 0.3                                             # oversample 강도 (0.3 ~ 0.5 권장)
CP_PER_IMAGE = 1                                        # 소수클래스 포함 이미지당 copy-paste 생성 수
PASTE_SCALE_RANGE = (0.9, 1.1)                          # 붙여넣기 스케일 랜덤 범위
PASTE_JITTER = 0.05                                     # 붙여넣기 좌표 지터 (이미지 크기 비율)
RAND_SEED = 42

random.seed(RAND_SEED)
np.random.seed(RAND_SEED)

IMG_EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.webp")

# =============================
# 유틸
# =============================
def safe_imread(path):
    """항상 BGR 3채널로 읽기. 실패하면 None."""
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    return img

def ensure_bgr3(img):
    """입력 이미지 채널을 BGR 3으로 통일."""
    if img is None:
        return None
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.ndim == 3 and img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
    return img

def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

def save_yaml(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        yaml.safe_dump(obj, f, sort_keys=False, allow_unicode=True)

def yolo_to_xyxy(box, W, H):
    cls, xc, yc, w, h = box
    x1 = (xc - w/2) * W
    y1 = (yc - h/2) * H
    x2 = (xc + w/2) * W
    y2 = (yc + h/2) * H
    return int(cls), max(0, int(x1)), max(0, int(y1)), min(W-1, int(x2)), min(H-1, int(y2))

def xyxy_to_yolo(cls, x1, y1, x2, y2, W, H):
    w = (x2 - x1) / W
    h = (y2 - y1) / H
    xc = (x1 + x2) / (2 * W)
    yc = (y1 + y2) / (2 * H)
    return [cls, xc, yc, w, h]

def read_labels(lbl_path):
    if not os.path.exists(lbl_path):
        return []
    with open(lbl_path, "r") as f:
        lines = f.read().strip().splitlines()
    boxes = []
    for ln in lines:
        if not ln.strip():
            continue
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        cls = int(parts[0]); xc, yc, w, h = map(float, parts[1:5])
        boxes.append([cls, xc, yc, w, h])
    return boxes

def write_labels(lbl_path, boxes):
    os.makedirs(os.path.dirname(lbl_path), exist_ok=True)
    with open(lbl_path, "w") as f:
        for b in boxes:
            f.write(f"{int(b[0])} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n")

def list_images(dir_or_txt):
    if os.path.isfile(dir_or_txt) and dir_or_txt.endswith(".txt"):
        return [p for p in open(dir_or_txt).read().splitlines() if p.strip()]
    paths = []
    for ex in IMG_EXTS:
        paths += glob.glob(os.path.join(dir_or_txt, "**", ex), recursive=True)
    return sorted(paths)

def infer_label_path(img_path, root_img, root_lbl):
    rel = os.path.relpath(img_path, root_img)
    lbl = os.path.splitext(rel)[0] + ".txt"
    return os.path.join(root_lbl, lbl)

def make_dir(p):
    os.makedirs(p, exist_ok=True)

def guess_roots(example_img):
    parts = example_img.split(os.sep)
    if "images" in parts:
        idx = parts.index("images")
        img_root = os.sep.join(parts[:idx+1])
        lbl_root = os.sep.join(parts[:idx] + ["labels"])
        return img_root, lbl_root
    return os.path.dirname(example_img), os.path.dirname(example_img)

# =============================
# 0) 데이터 로드
# =============================
cfg = load_yaml(DATA_YAML)
train_src = cfg["train"]
val_src = cfg.get("val", None)
names = cfg.get("names", None)

train_imgs = list_images(train_src)
if len(train_imgs) == 0:
    raise FileNotFoundError(f"train 경로에 이미지가 없습니다: {train_src}")

img_root, lbl_root = guess_roots(train_imgs[0])

# =============================
# 1) 분포 집계
# =============================
class_counts = Counter()
img_classes = {}

for img_path in tqdm(train_imgs, desc="Counting labels"):
    lbl_path = infer_label_path(img_path, img_root, lbl_root)
    if not os.path.exists(lbl_path):
        # 동일 경로 파일명.txt도 탐색
        alt = os.path.splitext(img_path)[0] + ".txt"
        lbl_path = alt if os.path.exists(alt) else lbl_path
    boxes = read_labels(lbl_path)
    cls_set = set()
    for b in boxes:
        class_counts[int(b[0])] += 1
        cls_set.add(int(b[0]))
    img_classes[img_path] = cls_set

if not class_counts:
    raise RuntimeError("라벨이 비어 있습니다. *.txt 라벨 파일을 확인하세요.")

max_count = max(class_counts.values())
minority = {c for c, k in class_counts.items() if k < 0.5 * max_count}

print("Class counts:", class_counts)
print("Minority classes:", minority)

# =============================
# 2) 오버샘플 대상 선정
# =============================
original_count = len(train_imgs)
target_count = math.ceil(original_count * 1.5)  # 1.5배 증강 목표
augmented_images_needed = max(0, target_count - original_count)

dup_list = []
for img_path, cls_set in img_classes.items():
    if not cls_set:
        dup = 1
    else:
        if cls_set & minority:
            min_freq = min(class_counts[c] for c in cls_set)
            ratio = max_count / max(min_freq, 1)
            dup = int(min(MAX_DUP, math.ceil(ratio ** ALPHA)))
        else:
            dup = 1
    dup = max(1, dup)
    dup_list += [img_path] * dup

dup_list = dup_list[:augmented_images_needed]
random.shuffle(dup_list)

# =============================
# 3) OUT_DIR로 원본 복사
# =============================
aug_img_dir = os.path.join(OUT_DIR, "images", "train")
aug_lbl_dir = os.path.join(OUT_DIR, "labels", "train")
make_dir(aug_img_dir); make_dir(aug_lbl_dir)

for img_path in tqdm(train_imgs, desc="Copy original to OUT_DIR"):
    rel = os.path.relpath(img_path, img_root) if "images" in img_path else os.path.basename(img_path)
    dst_img = os.path.join(aug_img_dir, rel)
    os.makedirs(os.path.dirname(dst_img), exist_ok=True)
    shutil.copy2(img_path, dst_img)

    lbl_src = infer_label_path(img_path, img_root, lbl_root)
    if not os.path.exists(lbl_src):
        lbl_src = os.path.splitext(img_path)[0] + ".txt"
    rel_lbl = os.path.relpath(lbl_src, lbl_root) if os.path.exists(lbl_src) and ("labels" in lbl_src) else os.path.basename(lbl_src)
    dst_lbl = os.path.join(aug_lbl_dir, rel_lbl)
    os.makedirs(os.path.dirname(dst_lbl), exist_ok=True)
    if os.path.exists(lbl_src):
        shutil.copy2(lbl_src, dst_lbl)
    else:
        open(dst_lbl, "w").close()

# copy-paste를 위한 배경 인덱스
all_bg_imgs = list_images(aug_img_dir)
bg_index = [p for p in all_bg_imgs]

# =============================
# 3-1) 안전한 paste 함수
# =============================
def paste_box(src_img, box_xyxy, dst_img, dst_boxes, cls_id):
    """경계/채널/크기 불일치를 완전히 방지하는 안전 paste."""
    src_img = ensure_bgr3(src_img)
    dst_img = ensure_bgr3(dst_img)
    if src_img is None or dst_img is None:
        return dst_img, dst_boxes

    Hd, Wd = dst_img.shape[:2]
    x1, y1, x2, y2 = box_xyxy

    # 유효 범위 클램프
    x1 = max(0, min(x1, src_img.shape[1]-1))
    x2 = max(0, min(x2, src_img.shape[1]-1))
    y1 = max(0, min(y1, src_img.shape[0]-1))
    y2 = max(0, min(y2, src_img.shape[0]-1))
    if x2 <= x1 or y2 <= y1:
        return dst_img, dst_boxes

    crop = src_img[y1:y2, x1:x2]
    crop = ensure_bgr3(crop)
    if crop is None or crop.size == 0:
        return dst_img, dst_boxes
    Hc, Wc = crop.shape[:2]
    if Hc < 2 or Wc < 2:
        return dst_img, dst_boxes

    # 스케일 (초기 크기)
    scale = random.uniform(*PASTE_SCALE_RANGE)
    init_w = max(2, int(Wc * scale))
    init_h = max(2, int(Hc * scale))

    # 위치 + 지터
    dx = random.randint(0, max(0, Wd - 1))
    dy = random.randint(0, max(0, Hd - 1))
    dx = int(min(max(0, dx + random.randint(int(-PASTE_JITTER*Wd), int(PASTE_JITTER*Wd))), Wd - 1))
    dy = int(min(max(0, dy + random.randint(int(-PASTE_JITTER*Hd), int(PASTE_JITTER*Hd))), Hd - 1))

    # 최종 붙여넣기 크기 (경계 고려)
    paste_w = min(init_w, Wd - dx)
    paste_h = min(init_h, Hd - dy)
    # 너무 작은/0이면 스킵
    if paste_w < 2 or paste_h < 2:
        return dst_img, dst_boxes
    # 너무 큰 패치(배경의 80% 이상)는 스킵(옵션)
    if paste_w > 0.8 * Wd or paste_h > 0.8 * Hd:
        return dst_img, dst_boxes

    # 최종 크기로 리사이즈 → 정확히 대입
    resized = cv2.resize(crop, (paste_w, paste_h), interpolation=cv2.INTER_LINEAR)
    if resized is None or resized.size == 0:
        return dst_img, dst_boxes
    if resized.shape[:2] != (paste_h, paste_w):
        return dst_img, dst_boxes

    dst_img[dy:dy+paste_h, dx:dx+paste_w] = resized

    # 라벨 추가
    dst_boxes.append(xyxy_to_yolo(cls_id, dx, dy, dx + paste_w, dy + paste_h, Wd, Hd))
    return dst_img, dst_boxes

# =============================
# 3-2) Copy-Paste 생성
# =============================
print("Generating copy-paste aug...")

for img_path in tqdm(train_imgs, desc="Copy-Paste"):
    # 원본 라벨
    lbl_src = infer_label_path(img_path, img_root, lbl_root)
    if not os.path.exists(lbl_src):
        lbl_src = os.path.splitext(img_path)[0] + ".txt"
    boxes = read_labels(lbl_src)

    # 소스 이미지
    src_img = safe_imread(img_path)
    src_img = ensure_bgr3(src_img)
    if src_img is None:
        continue
    H, W = src_img.shape[:2]

    # 소수 클래스 박스만 수집 (너무 작은 박스 제외)
    min_boxes_xyxy = []
    for b in boxes:
        cls = int(b[0])
        if cls in minority:
            _, x1, y1, x2, y2 = yolo_to_xyxy(b, W, H)
            if (x2 - x1) >= 4 and (y2 - y1) >= 4:
                min_boxes_xyxy.append((cls, (x1, y1, x2, y2)))

    if not min_boxes_xyxy:
        continue

    for n in range(CP_PER_IMAGE):
        bg_img_path = random.choice(bg_index)
        bg = safe_imread(bg_img_path)
        bg = ensure_bgr3(bg)
        if bg is None:
            continue

        Hd, Wd = bg.shape[:2]
        bg_lbl_rel = os.path.relpath(bg_img_path, aug_img_dir)
        bg_lbl_path = os.path.join(aug_lbl_dir, os.path.splitext(bg_lbl_rel)[0] + ".txt")
        bg_boxes = read_labels(bg_lbl_path)

        num_paste = random.randint(1, min(3, len(min_boxes_xyxy)))
        paste_samples = random.sample(min_boxes_xyxy, num_paste)
        new_img = bg.copy()
        new_boxes = bg_boxes.copy()

        for cls_id, (x1, y1, x2, y2) in paste_samples:
            new_img, new_boxes = paste_box(src_img, (x1, y1, x2, y2), new_img, new_boxes, cls_id)

        # 아웃 경로
        base, ext = os.path.splitext(bg_lbl_rel)
        out_stub = f"{base}_cp{n:02d}"
        out_img_path = os.path.join(aug_img_dir, out_stub + ".jpg")
        out_lbl_path = os.path.join(aug_lbl_dir, out_stub + ".txt")
        os.makedirs(os.path.dirname(out_img_path), exist_ok=True)
        cv2.imwrite(out_img_path, new_img)
        write_labels(out_lbl_path, new_boxes)

# =============================
# 4) 오버샘플링된 train.txt 저장
# =============================
all_train_imgs_after_aug = list_images(aug_img_dir)

new_img_classes = {}
new_counts = Counter()
for img_path in tqdm(all_train_imgs_after_aug, desc="Recount after aug"):
    rel = os.path.relpath(img_path, aug_img_dir)
    lbl_path = os.path.join(aug_lbl_dir, os.path.splitext(rel)[0] + ".txt")
    boxes = read_labels(lbl_path)
    cls_set = set()
    for b in boxes:
        new_counts[int(b[0])] += 1
        cls_set.add(int(b[0]))
    new_img_classes[img_path] = cls_set

max_count2 = max(new_counts.values()) if new_counts else 1

oversampled_list = []
for img_path, cls_set in new_img_classes.items():
    if not cls_set:
        dup = 1
    else:
        min_freq = min(new_counts[c] for c in cls_set) if cls_set else max_count2
        ratio = max_count2 / max(min_freq, 1)
        dup = int(min(MAX_DUP, math.ceil(ratio ** ALPHA)))
    dup = max(1, dup)
    oversampled_list += [img_path] * dup

random.shuffle(oversampled_list)

train_txt_path = os.path.join(OUT_DIR, "train.txt")
with open(train_txt_path, "w") as f:
    for p in oversampled_list:
        f.write(p + "\n")

print(f"Saved oversampled train list: {train_txt_path} ({len(oversampled_list)} lines)")

# =============================
# 5) 새 data.yaml 저장
# =============================
new_yaml = dict(cfg)  # shallow copy
new_yaml["train"] = train_txt_path
NEW_DATA_YAML = os.path.join(OUT_DIR, "data_oversampled.yaml")
save_yaml(new_yaml, NEW_DATA_YAML)
print(f"Saved new data.yaml: {NEW_DATA_YAML}")


Counting labels: 100%|██████████| 5131/5131 [00:00<00:00, 12360.85it/s]


Class counts: Counter({0: 10174, 4: 4564, 5: 2069, 3: 1636, 1: 1429, 2: 415})
Minority classes: {1, 2, 3, 4, 5}


Copy original to OUT_DIR: 100%|██████████| 5131/5131 [00:15<00:00, 335.82it/s]


Generating copy-paste aug...


Recount after aug: 100%|██████████| 9611/9611 [00:01<00:00, 7432.43it/s]

Saved oversampled train list: /content/oversampled_dataset/train.txt (9611 lines)
Saved new data.yaml: /content/oversampled_dataset/data_oversampled.yaml


In [ ]:
import os, json, yaml
from pathlib import Path
from ultralytics import RTDETR

# === 경로 설정 ===
DATA_ROOT = Path("oversampled_dataset")   # YOLO 포맷 루트
VAL_DIRNAME = "val"               # 'val'이면 여기만 'val'로

DATA_YAML = DATA_ROOT / "data_oversampled.yaml"  # YOLO data.yaml (names 포함)

def write_dataset_yaml(path="dataset.yaml"):
    ds = {
        "train": "/content/oversampled_dataset/train.txt",
        "val":   "/content/split_dataset_merged/val/images",
        "test":  "/content/split_dataset_merged/test/images",
        "names": ["car","car_door","hand","human","license_plate","trash"],
    }
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(ds, f, sort_keys=False, allow_unicode=True)
    print(f"✅ wrote {path}")
    return path


def main():
    yaml_path = write_dataset_yaml("dataset.yaml")

    # 모델 로드
    model = RTDETR("rtdetr-l.pt")

    # 메모리 안전 설정 (T4 기준)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    model.train(
        data=yaml_path,
        epochs=150,
        imgsz=640,      # T4에서 안전
        batch=16,        # 부족하면 1, 여유면 2
        device=0,
        workers=2,
        rect=True,
        project="runs",
        name="rtdetr-custom",
        pretrained=True,
        lr0=2e-4,
        weight_decay=5e-2,
        patience=10,
    )

    # 테스트도 같은 해상도로
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=2
    )
    print("📊 Test metrics:", test_metrics)

if __name__ == "__main__":
    main()


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ wrote dataset.yaml


Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=rtdetr-custom, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, p

Overriding model.yaml nc=80 with nc=6
WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1     25248  ultralytics.nn.modules.block.HGStem          [3, 32, 48]                   
  1                  -1  6    155072  ultralytics.nn.modules.block.HGBlock         [48, 48, 128, 3, 6]           
  2                  -1  1      1408  ultralytics.nn.modules.conv.DWConv           [128, 128, 3, 2, 1, False]    
  3                  -1  6    839296  ultralytics.nn.modules.block.HGBlock         [128, 96, 512, 3, 6]          
  4                  -1  1      5632  ultralytics.nn.modules.conv.DWConv           [512, 512, 3, 2, 1, False]    
  5                  -1  6   1695360  ultralytics.nn.modules.block.HGBlock         [512, 192, 1024, 5, 6, True, False]
  6                  -1  6   2055808  ultralytics.nn.modules.block.HGBlock         [1024, 192, 1024, 5, 

 16                  -1  3   2232320  ultralytics.nn.modules.block.RepC3           [512, 256, 3]                 
 17                  -1  1     66048  ultralytics.nn.modules.conv.Conv             [256, 256, 1, 1]              
 18                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 19                   3  1    131584  ultralytics.nn.modules.conv.Conv             [512, 256, 1, 1, None, 1, 1, False]
 20            [-2, -1]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 21                  -1  3   2232320  ultralytics.nn.modules.block.RepC3           [512, 256, 3]                 
 22                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
 23            [-1, 17]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 24                  -1  3   2232320  ultralytics.nn.modules.block.RepC3           

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3274.3±663.5 MB/s, size: 336.6 KB)


...

                   all       1412       4816      0.356     0.0518      0.067     0.0224

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/492 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/autograd/graph.py:823: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:92.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
      4/150      11.7G      1.439     0.3611     0.2711          3        640: 100%|██████████| 492/492 [09:25<00:00,  1.15s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 45/45 [00:28<00:00,  1.59it/s]


                   all       1412       4816      0.409     0.0724     0.0631     0.0238

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


  0%|          | 0/492 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/autograd/graph.py:823: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:92.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
      5/150      11.6G      1.489     0.3557     0.3079        112        640:  27%|██▋       | 135/492 [02:36<06:53,  1.16s/it]


KeyboardInterrupt: 

In [ ]:
import shutil
from google.colab import files

# 1. 폴더를 ZIP으로 압축
shutil.make_archive('runs', 'zip', 'runs')

# 2. 다운로드 링크 제공
files.download('runs.zip')


In [ ]:
import os, yaml
from pathlib import Path
from ultralytics import RTDETR

# === 경로 설정 ===
DATA_ROOT = Path("oversampled_dataset")
DATA_YAML = DATA_ROOT / "data_oversampled.yaml"  # (미사용이면 유지/삭제 아무거나 OK)

def write_dataset_yaml(path="dataset.yaml"):
    ds = {
        "train": "/content/oversampled_dataset/train.txt",
        "val":   "/content/split_dataset_merged/val/images",
        "test":  "/content/split_dataset_merged/test/images",
        "names": ["car","car_door","hand","human","license_plate","trash"],
    }
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(ds, f, sort_keys=False, allow_unicode=True)
    print(f"✅ wrote {path}")
    return path

def main():
    yaml_path = write_dataset_yaml("dataset.yaml")

    # 모델 로드
    model = RTDETR("rtdetr-l.pt")

    # 메모리 안전 설정 (T4)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    # ======= 개선된 훈련 설정 =======
    model.train(
        data=yaml_path,
        epochs= 150,                 # 50→150: RT-DETR-L은 장에폭에서 한 번 더 뜁니다
        imgsz=640,
        batch=16,
        device=0,
        workers=8,                  # 2→8: I/O 병목 완화
        rect=False,                 # True→False: 모자이크/스케일 효과 회복
        close_mosaic=20,            # 후반부 안정화
        project="runs",
        name="rtdetr-custom-improved",
        pretrained=True,

        # === 최적화/스케줄 ===
        optimizer="AdamW",          # 명시! auto가 lr0/momentum 무시하는 것 방지
        lr0=0.0005,                 # 0.001→0.0005: 살짝 보수적으로
        lrf=0.01,
        weight_decay=0.01,          # 0.05→0.01: 완화
        momentum=0.9,               # AdamW의 beta2에 해당(ultralytics 내부 매핑)
        cos_lr=True,
        warmup_epochs=5.0,
        warmup_bias_lr=0.1,
        warmup_momentum=0.8,

        # === 조기 종료/저장 ===
        patience=10,                # 15→50: 너무 일찍 멈추는 것 방지
        save_period=5,

        # === AMP/결정성 ===
        amp=True,
        deterministic=False,

        # === 증강 (강도 완화) ===
        auto_augment=None,          # randaugment 해제
        erasing=0.15,               # 0.4→0.15
        hsv_h=0.015,
        hsv_s=0.5,                  # 0.7→0.5
        hsv_v=0.4,
        degrees=0.0,
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        copy_paste=0.0,
    )

    # ======= 검증/테스트 (운영 포인트 재탐색 + 리포트 저장) =======
    # 검증 세트
    _ = model.val(
        data=yaml_path,
        split="val",
        imgsz=640,
        batch=16,
        device=0,
        workers=8,
        iou=0.6,            # 0.7→0.6: 리콜 개선 포인트 관찰
        plots=True,         # PR/F1/혼동행렬 등 저장
        verbose=True
    )

    # 테스트 세트
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        imgsz=640,
        batch=16,
        device=0,
        workers=8,
        iou=0.6,
        plots=True,
        verbose=True
    )
    print("📊 Test metrics:", test_metrics)

if __name__ == "__main__":
    main()


✅ wrote dataset.yaml
Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.15, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.9, mosaic=1.0, multi_scale=False, name=rtdetr-custom-improved3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=

train: Scanning /content/oversampled_dataset/labels/train.cache... 7857 images, 6 backgrounds, 0 corrupt: 100%|██████████| 7857/7857 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 260, len(boxes) = 34806. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 946.0±742.6 MB/s, size: 314.7 KB)


val: Scanning /content/split_dataset_merged/val/labels.cache... 1412 images, 2 backgrounds, 0 corrupt: 100%|██████████| 1412/1412 [00:00<?, ?it/s]

...

                   all       1412       4816      0.734      0.689      0.683      0.327
                   car       1393       2030      0.904      0.974      0.966      0.673
              car_door        460        462      0.877      0.909      0.925       0.49
                  hand        101        109      0.471      0.147      0.165     0.0347
                 human        489        534      0.714      0.683      0.718      0.281
         license_plate        823        983      0.659      0.789      0.689      0.263
                 trash        685        698      0.778       0.63      0.636       0.22
Speed: 0.8ms preprocess, 33.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to runs/rtdetr-custom-improved32
Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
rt-detr-l summary: 302 layers, 31,996,070 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 60.2±21.4 MB/s, size: 251.1 KB)


val: Scanning /content/split_dataset_merged/test/labels... 825 images, 0 backgrounds, 0 corrupt: 100%|██████████| 825/825 [00:02<00:00, 393.53it/s]

val: New cache created: /content/split_dataset_merged/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 4, len(boxes) = 2996. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 52/52 [00:32<00:00,  1.60it/s]


                   all        825       2996       0.62      0.622      0.592      0.281
                   car        825       1423      0.921      0.957      0.967      0.658
              car_door        239        243      0.798      0.905      0.879      0.444
                  hand         26         36          0          0     0.0209    0.00565
                 human        236        254      0.726      0.618      0.635       0.25
         license_plate        450        650      0.622      0.671       0.57      0.188
                 trash        390        390      0.653      0.578      0.479      0.141
Speed: 0.3ms preprocess, 33.9ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/rtdetr-custom-improved33
📊 Test metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cb31ac52

In [ ]:
import shutil
from google.colab import files

# 1. 폴더를 ZIP으로 압축
shutil.make_archive('runs2', 'zip', 'runs')

# 2. 다운로드 링크 제공
files.download('runs2.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import shutil


# 2. 복사할 파일 목록
files = [
    #"v11_video_frame_output_grouped.zip",
    #"v11_video_frame_output_grouped2.zip",
    "runs2.zip",
    #"littering_frames_grouped_0814.zip"
]

# 3. 드라이브 저장 경로 (예: MyDrive/ColabOutputs 폴더)
save_path = "/content/drive/MyDrive/ColabOutputs/"

# 폴더 없으면 생성
import os
os.makedirs(save_path, exist_ok=True)

# 4. 파일 복사
for f in files:
    shutil.copy(f, save_path)

print("✅ 모든 파일이 Google Drive에 저장되었습니다!")


✅ 모든 파일이 Google Drive에 저장되었습니다!


In [ ]:
from ultralytics import YOLO

# 학습 완료된 best 모델 로드
model = YOLO("/content/best_yolo.pt")

# test set으로 평가
test_metrics = model.val(
    data= '/content/split_dataset_merged/data.yaml',   # data.yaml 경로
    split="test",     # validation이 아니라 test split 사용
    imgsz=640,        # 해상도
    batch=16,         # 배치 크기
    device='cpu',         # GPU 장치 (0번)
    workers=2         # dataloader workers
)

print("📊 Test metrics:", test_metrics)


Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
YOLO11l summary (fused): 190 layers, 25,283,938 parameters, 0 gradients, 86.6 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 73.4±20.5 MB/s, size: 251.3 KB)



val: Scanning /content/split_dataset_merged/test/labels... 723 images, 1 backgrounds, 0 corrupt: 100%|██████████| 723/723 [00:02<00:00, 356.89it/s]

val: New cache created: /content/split_dataset_merged/test/labels.cache
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 26, len(boxes) = 3328. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 46/46 [20:05<00:00, 26.20s/it]


                   all        723       3328      0.727      0.628      0.651      0.321
                   car        710       1819      0.896      0.965      0.972       0.69
              car_door         99         99      0.838      0.818      0.862      0.414
                  hand         63         67      0.519      0.343      0.316     0.0956
                 human        203        204       0.76      0.819      0.841       0.37
         license_plate        463        803      0.623      0.383      0.429       0.18
                 trash        328        336      0.725      0.443      0.486      0.174
Speed: 1.2ms preprocess, 1642.2ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to runs/detect/val3
📊 Test metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x798b0c415490>
curves

In [ ]:
from ultralytics import RTDETR

# 학습 완료된 best 모델 로드
model = RTDETR("/content/best_detr.pt")

# test set으로 평가
test_metrics = model.val(
    data= '/content/split_dataset_merged/data.yaml',   # data.yaml 경로
    split="test",     # validation이 아니라 test split 사용
    imgsz=640,        # 해상도
    batch=16,         # 배치 크기
    device='cpu',         # GPU 장치 (0번)
    workers=2         # dataloader workers
)

print("📊 Test metrics:", test_metrics)


Ultralytics 8.3.179 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
rt-detr-l summary: 302 layers, 31,996,070 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1858.6±370.1 MB/s, size: 336.2 KB)


val: Scanning /content/split_dataset_merged/test/labels.cache... 723 images, 1 backgrounds, 0 corrupt: 100%|██████████| 723/723 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 26, len(boxes) = 3328. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 46/46 [42:32<00:00, 55.48s/it]


                   all        723       3328      0.759      0.777      0.774      0.357
                   car        710       1819      0.893      0.966      0.964      0.663
              car_door         99         99      0.778      0.909      0.884      0.443
                  hand         63         67      0.683      0.478      0.495      0.114
                 human        203        204      0.841      0.907      0.931      0.417
         license_plate        463        803      0.647      0.743      0.725      0.294
                 trash        328        336      0.711      0.661      0.643      0.207
Speed: 7.5ms preprocess, 3496.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/detect/val4
📊 Test metrics: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x798b05597b10>
curves